# Train 3-branch model (1x3D ResNeXt + 2x2D MaxViT + CrossGate) + optional information-theory survey

Canonical training notebook for the repo's recommended main model: one 3D ResNeXt branch over the OCT volume
(96^3, resized on the fly from the raw 200^3 storage) plus two 2D MaxViT-Tiny branches over the en-face views
(`slab_mip` + `aip_full`, projected from the raw 200^3 volume), fused by CrossGate and classified by a linear
head. Architecture lives in `scripts/final_model.py`; the resumable trainer, metrics, calibration, reporting
and X-AI live in `scripts/final_training.py`; the optional analysis lives in `scripts/information_theory.py`.

- `N_2D=2` gives the 3-branch model (default); `N_2D=1` gives the 1x3D + slab_mip ablation.
- `RUN_XAI=True` explains the best model (Grad-CAM 3D/2D, occlusion, integrated gradients, fusion attention,
  branch drop) and logs `xai/fusion_table` + `xai/*` to W&B.
- `RUN_INFO=True` additionally runs the deep information-theory survey: branch and fused embeddings,
  probe-based branch ablation, multi-seed DV/NWJ/InfoNCE MI with median/negative-rate diagnostics, MIC,
  surrogate permutation testing, joint/conditional MI, redundancy-vs-synergy interaction proxies and the
  information plane. Results are logged as `info_theory.*` + W&B tables (`report/*_table`); per-epoch
  embeddings are captured only when this flag is on. Negative estimates are kept as-is (never clamped).

Storage resolution is config-driven (`STORE_RES`, default raw 200^3); downloads always use authenticated
`HF_TOKEN` with `allow_patterns` for the declared splits. Logging follows `docs/notebook-conventions.md`: train
metrics every optimizer step, val + test metrics every epoch, calibrated `train/val/test` + bootstrap CI +
`report/split_table` in W&B, and every artifact local + Drive.

`TB3_SMOKE=1` runs tiny synthetic CPU data with offline W&B. Real runs need a GPU plus `WANDB_API_KEY` and `HF_TOKEN`.

In [ ]:
import os, sys, subprocess
from pathlib import Path
SMOKE = os.environ.get('TB3_SMOKE', '0') == '1'
if not SMOKE and 'google.colab' in sys.modules:
    repo = Path('/content/glaucoma-thesis')
    if not repo.exists():
        subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/Tqhuyen/glaucoma-thesis.git', str(repo)], check=True)
    os.chdir(repo)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'timm', 'wandb', 'scipy', 'matplotlib', 'huggingface_hub', 'python-dotenv', 'hf-transfer'], check=True)
    os.environ.setdefault('HF_HUB_ENABLE_HF_TRANSFER', '1')
sys.path.insert(0, str(Path.cwd()))
import json, random, tempfile, time
import numpy as np
import torch
import torch.nn.functional as F
import wandb
import matplotlib
matplotlib.use('Agg')
from scripts import final_model as fm, final_training as ft, information_theory as it
ft.load_env_file()
DEVICE = torch.device('cpu' if SMOKE else ('cuda' if torch.cuda.is_available() else 'cpu'))
if DEVICE.type == 'cuda':
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
if SMOKE:
    torch.set_num_threads(1)


## Configuration
`STORE_RES` is the on-disk storage resolution (200/128/96/...); `RES3D` is the 3D model input (resized on the
fly) and `RES2D` the 2D view cache size. Defaults store raw 200^3 (`harvard-oct-glaucoma-200`) but train the 3D
branch at 96^3 (`RES3D=96`), with 2D views projected from the raw volume; change `STORE_RES`/`RES3D` and
`HF_DATA_REPO` together to switch storage. `RUN_XAI` and `RUN_INFO` gate the optional stages in this same
notebook (no forks). `IT_ESTIMATOR_METHODS`/`IT_ESTIMATOR_SEEDS` control the multi-seed MI battery;
`IT_SURROGATES` (fast MIC null) and `IT_MINE_SURROGATES` (MINE null) set permutation counts - use >=1000
for thesis-level p-value resolution. Change `RUN_GROUP` for every new config; `RESUME=True` requires an identical training
config (except `epochs`), and `WARM_START_WEIGHTS` can transfer a 200^3-trained model (fully convolutional
encoder).

Model init order: resume checkpoint (`last.pt`) > existing weights at `WARM_START_WEIGHTS`/`best_weights.pt`
(loaded into the model, pretrained backbone skipped) > only when no weights exist is a new model initialized
with a pretrained 2D backbone.

Training extension: set `RESUME=True` and `EXTEND_EPOCHS=N` and re-run the training cell to add N epochs to
the existing run (other config must match; `epochs` is the only allowed difference). Patience resets and the
LR follows the new total cosine schedule.

In [ ]:
RUN_GROUP = 'crossgate_3branch_s42'
RUN_TARGET = 'raw_s42'
RESUME = False
EXTEND_EPOCHS = 0
WARM_START_WEIGHTS = ''
CHECKPOINT_EVERY_STEPS = 10
DATASETS = ['raw']
SEEDS = [42]
N_2D = 1 if SMOKE else 2
STORE_RES = 8 if SMOKE else 200
RES3D = 8 if SMOKE else 96
RES2D = 8 if SMOKE else 224
D_LATENT, ENC2D = 256, 'maxvit_tiny_rw_224'
ENC3D_FEATURES = (32, 64, 128, 192)
EPOCHS, BS, GRAD_ACCUM = (1, 2, 2) if SMOKE else (10, 2, 8)
AUTO_BATCH = not SMOKE
TARGET_VRAM_GB = float(os.environ.get('TB3_TARGET_VRAM_GB', '60'))
EFFECTIVE_BATCH = 16
NUM_WORKERS = 0 if os.name == 'nt' else 4
LR, WD, PATIENCE = 1e-4, 1e-4, 4
RUN_XAI = True
RUN_INFO = True
IT_MINE_STEPS = 40 if SMOKE else 600
IT_MINE_HIDDEN = 32 if SMOKE else 64
IT_PROBE_STEPS = 60 if SMOKE else 400
IT_ESTIMATOR_METHODS = ('dv', 'nwj', 'infonce')
IT_ESTIMATOR_SEEDS = (0, 1) if SMOKE else (0, 1, 2, 3, 4)
IT_SURROGATES = 3 if SMOKE else 1000
IT_MINE_SURROGATES = 3 if SMOKE else 200
IT_PCA_DIM = 6 if SMOKE else 32
IT_INPUT_RES = 4 if SMOKE else 16
IT_PLANE_FOLDS = 2 if SMOKE else 3
IT_MIC_PCS = 4 if SMOKE else 8
HF_DATA_REPO = os.environ.get('HF_DATA_REPO', 'tqhuyen/harvard-oct-glaucoma-200')
SPLITS = ('Training', 'Validation', 'Test')
HF_DATA_PATTERNS = [f'{split}_{kind}.npy' for split in SPLITS for kind in ('volumes', 'labels')]
SMOKE_ROOT = Path(tempfile.mkdtemp(prefix='tb3_smoke_')) if SMOKE else None
DATA_ROOT = SMOKE_ROOT / 'data' if SMOKE else Path('/content/final_data')
LOCAL_ROOT = SMOKE_ROOT / 'runs' if SMOKE else Path('outputs/crossgate_3branch') / RUN_GROUP
DRIVE_MOUNT = Path(os.environ.get('DRIVE_MOUNT', '/content/drive'))
DRIVE_ROOT = Path(os.environ.get('DRIVE_ROOT', '/content/drive/MyDrive/MasterBKDN/Thesis'))
DRIVE_DIR = DRIVE_ROOT / 'crossgate_3branch' / RUN_GROUP
selected = [(ds, seed, f'{ds}_s{seed}') for ds in DATASETS for seed in SEEDS if not RUN_TARGET or RUN_TARGET == f'{ds}_s{seed}']
if not selected:
    raise ValueError('RUN_TARGET does not match DATASETS/SEEDS')
if N_2D not in (1, 2):
    raise ValueError('N_2D must be 1 (ablation) or 2 (3-branch model)')
if RES3D > STORE_RES:
    raise ValueError('RES3D cannot exceed the stored resolution')
if WARM_START_WEIGHTS and (RESUME or len(selected) != 1):
    raise ValueError('Warm-start requires RESUME=False and one explicitly selected RUN_TARGET')
if not SMOKE:
    if not os.path.ismount(DRIVE_MOUNT):
        from google.colab import drive
        drive.mount(str(DRIVE_MOUNT))
    if not os.path.ismount(DRIVE_MOUNT) or not DRIVE_ROOT.resolve().is_relative_to(DRIVE_MOUNT.resolve()):
        raise RuntimeError('Drive must be a verified mounted filesystem, not a local directory')
    DRIVE_DIR.mkdir(parents=True, exist_ok=True)
if WARM_START_WEIGHTS and not Path(WARM_START_WEIGHTS).is_file():
    raise FileNotFoundError(WARM_START_WEIGHTS)
DATA_ROOT.mkdir(parents=True, exist_ok=True)
STORAGE = ft.Artifacts(LOCAL_ROOT, None if SMOKE else DRIVE_DIR, smoke=SMOKE)
DATA_STORAGE = ft.Artifacts(DATA_ROOT, None if SMOKE else DRIVE_DIR / 'data', smoke=SMOKE)
print('Device:', DEVICE, '| branches: 3D +', N_2D, 'x 2D | storage', STORE_RES, '| res3d', RES3D, '| res2d', RES2D, '| xai', RUN_XAI, '| info', RUN_INFO)


## Data
D1 downloads only the declared splits from HF with `HF_TOKEN` (`allow_patterns`); D2 verifies the raw
storage; D3 builds/caches the 2D views + depth-axis once per split; D4 defines the dataset factory used by
the training loop.

### D1 - Download (HF auth + allow_patterns)


In [ ]:
if SMOKE:
    rng = np.random.default_rng(0)
    for split, n in zip(SPLITS, (10, 6, 6)):
        np.save(DATA_ROOT / f'{split}_volumes.npy', rng.integers(0, 255, (n, 1, STORE_RES, STORE_RES, STORE_RES), dtype=np.uint8))
        np.save(DATA_ROOT / f'{split}_labels.npy', np.arange(n, dtype=np.int64) % 2)
    print('[data] smoke synthetic arrays ready at', DATA_ROOT)
else:
    from huggingface_hub import snapshot_download
    token = os.environ.get('HF_TOKEN')
    if not token:
        try:
            from google.colab import userdata
            token = userdata.get('HF_TOKEN')
        except Exception:
            token = None
    if not token:
        raise RuntimeError('Authenticated HF download requires HF_TOKEN in .env/environment or Colab Secrets')
    print('[data] repo', HF_DATA_REPO, '| patterns', len(HF_DATA_PATTERNS))
    if not all((DATA_ROOT / name).is_file() for name in HF_DATA_PATTERNS):
        print('[data] downloading declared splits (authenticated)...')
        snapshot_download(repo_id=HF_DATA_REPO, repo_type='dataset', local_dir=str(DATA_ROOT), token=token, allow_patterns=HF_DATA_PATTERNS)
        print('[data] download complete')
    else:
        print('[data] cache hit; no download needed')

### D2 - Verify raw storage


In [ ]:
for split in SPLITS:
    volumes = np.load(DATA_ROOT / f'{split}_volumes.npy', mmap_mode='r')
    labels = np.load(DATA_ROOT / f'{split}_labels.npy')
    if not SMOKE and tuple(volumes.shape[-3:]) != (STORE_RES, STORE_RES, STORE_RES):
        raise ValueError(f'Real training requires STORE_RES-cubed storage, got {volumes.shape[-3:]}')
    if len(volumes) != len(labels):
        raise ValueError(f'{split}: volume/label count mismatch')
    print(f'[storage] {split}: {tuple(volumes.shape)} | labels {len(labels)} | pos {int(np.asarray(labels).sum())}')

### D3 - View + depth-axis cache


In [ ]:
for split in SPLITS:
    views_path, depth_path = ft.build_views(DATA_ROOT / f'{split}_volumes.npy', res2d=RES2D)
    print(f'[views] {split}: {views_path.name} + {depth_path.name}')
for split in SPLITS:
    for path in DATA_ROOT.glob(f'{split}_volumes_*{RES2D}*'):
        if '.partial.' not in path.name:
            DATA_STORAGE.sync(path)
print('[cache] view/depth caches synced')

### D4 - Dataset factory


In [ ]:
def make_datasets(ds, seed):
    datasets = [ft.FinalDataset(DATA_ROOT / f'{s}_volumes.npy', DATA_ROOT / f'{s}_labels.npy', res3d=RES3D, res2d=RES2D, seed=seed, train=s == 'Training') for s in SPLITS]
    if ds != 'raw':
        raise ValueError('This notebook trains on raw storage; use the final crossgate notebook for denoised runs')
    for split, dataset in zip(SPLITS, datasets):
        print(f'[dataset] {split}: n={len(dataset)} pos={int(np.asarray(dataset.labels).sum())} | res3d={RES3D} | views {RES2D}px | {dataset.source.name}')
    return datasets

## Train, evaluate, persist and explain
The training loop uses small factories: model init (load from path first), batch/config resolution
(VRAM probe), val/test callbacks, and report/checkpoint/X-AI finalization. Metrics cadence: train per
optimizer step, val + test per epoch; calibrated `train/val/test` + CI + `report/split_table` at the end.

### T1 - Model (load from path first)


In [ ]:
def make_model(tag_resume, tag_warm_start, artifacts):
    model_path = Path(tag_warm_start) if tag_warm_start else artifacts.local / 'best_weights.pt'
    if tag_resume or not model_path.is_file():
        model_path = None
    model = (ft.SmokeModel() if SMOKE else fm.FinalModel(n_2d=N_2D, D=D_LATENT, enc2d=ENC2D, enc3d_features=ENC3D_FEATURES, enc2d_pretrained=not (tag_resume or bool(tag_warm_start) or model_path is not None))).to(DEVICE)
    if model_path is not None:
        ft.load_weights(model, model_path)
        print('[model] loaded weights from', model_path)
    elif tag_resume:
        print('[model] resume: weights restored from last.pt by Trainer')
    else:
        print('[model] no existing weights; initialized a new model')
    return model

### T2 - Batch probe + config


In [ ]:
def resolve_batch(tag, tr):
    if RESUME:
        saved = ft.saved_config(LOCAL_ROOT / tag, DRIVE_DIR / tag)
        if saved:
            print('[batch] resume reuse bs/accum', int(saved['batch_size']), int(saved['grad_accum']))
            return int(saved['batch_size']), int(saved['grad_accum']), None
    if AUTO_BATCH and DEVICE.type == 'cuda':
        probe = ft.find_batch_size(lambda: ft.SmokeModel() if SMOKE else fm.FinalModel(n_2d=N_2D, D=D_LATENT, enc2d=ENC2D, enc3d_features=ENC3D_FEATURES, enc2d_pretrained=False), tr, device=DEVICE, start=BS, target_gb=TARGET_VRAM_GB, num_workers=NUM_WORKERS)
        bs = int(probe['batch_size'])
        print('[batch] probe result:', probe)
        return bs, max(1, round(EFFECTIVE_BATCH / bs)), probe
    return BS, GRAD_ACCUM, None

def make_config(ds, seed, tr, va, te, bs, accum):
    ytr = tr.labels
    if set(np.unique(ytr)) != {0, 1}:
        raise ValueError('Training requires both binary classes')
    weights = [len(ytr) / (2 * int((ytr == c).sum())) for c in (0, 1)]
    return dict(dataset=ds, seed=seed, epochs=EPOCHS, batch_size=bs, grad_accum=accum, lr=LR, weight_decay=WD, patience=PATIENCE, checkpoint_steps=CHECKPOINT_EVERY_STEPS, class_weights=weights, res3d=RES3D, res2d=RES2D, store_res=STORE_RES, n2d=N_2D, latent=D_LATENT, enc2d=ENC2D, enc3d_features=list(ENC3D_FEATURES), smoke=SMOKE, torch_version=str(torch.__version__), device=str(DEVICE), data=[ft.data_identity(p) for d in (tr, va, te) for p in (d.source, d.label_path)])

def log_batch_probe(run, probe, bs, accum):
    if not probe:
        return
    table = wandb.Table(columns=['batch_size', 'peak_gb', 'ok'])
    for trial in probe['trials']:
        table.add_data(trial['batch_size'], trial['peak_gb'], trial['ok'])
    run.log({'report/batch_probe_table': table})
    run.summary.update({'batch/size': bs, 'batch/accum': accum, 'batch/peak_gb': probe['peak_gb'], 'batch/target_gb': TARGET_VRAM_GB})

### E1/E2 - Val/test callbacks (per-epoch metrics)


In [ ]:
def make_val_eval(va, bs, embeddings):
    def evaluate(model):
        if RUN_INFO:
            capture = it.collect_embeddings(model, va, bs, input_res=IT_INPUT_RES, num_workers=NUM_WORKERS)
            embeddings['z'].append(capture['z'])
            embeddings['e3d'].append(capture['e3d'])
            embeddings['e2d'].append(capture['e2d'])
            embeddings['y'].append(capture['y'])
            if not embeddings['x']:
                embeddings['x'].append(capture['x'])
            return {**fm.full_metrics(capture['probs'], capture['y']), 'loss': float(F.cross_entropy(torch.tensor(capture['logits']), torch.tensor(capture['y'])))}
        p, y, logits = ft.predict(model, va, bs, num_workers=NUM_WORKERS)
        return {**fm.full_metrics(p, y), 'loss': float(F.cross_entropy(torch.tensor(logits), torch.tensor(y)))}
    return evaluate

def make_test_eval(te, bs):
    def evaluate_test(model):
        p, y, logits = ft.predict(model, te, bs, num_workers=NUM_WORKERS)
        return {**fm.full_metrics(p, y), 'loss': float(F.cross_entropy(torch.tensor(logits), torch.tensor(y)))}
    return evaluate_test

### E3 - Calibrated report + checkpoint + X-AI


In [ ]:
def finish_tag(model, trainer, tr, va, te, artifacts, run, tag, seed, started, bs):
    res, probs, labels = ft.calibrated_report(model, va, te, bs, smoke=SMOKE, train=tr, num_workers=NUM_WORKERS)
    res.update(tag=tag, seed=seed, hist=trainer.history, minutes=round((time.time() - started) / 60, 2))
    print(f'[eval] test AUC={res["test"]["auc_roc"]:.4f} F1={res["test"]["f1"]:.4f} | val AUC={res["val"]["auc_roc"]:.4f}')
    ft.log_report(run, res)
    params = sum(p.numel() for p in model.parameters())
    run.summary.update({'threshold': res['threshold'], 'temperature': res['temperature'], 'params': params, 'minutes': res['minutes']})
    weights_path = artifacts.save(ft.cpu_state(model), 'best_weights.pt')
    print('[checkpoint] saved', weights_path.name)
    ft.save_report(res, probs, labels, artifacts, run)
    if RUN_XAI:
        ft.save_xai(model, va, artifacts, run, smoke=SMOKE)
    return dict(res=res, weights_path=str(weights_path), params=params, test_probs=probs.tolist(), test_labels=labels.tolist())

### T3 - Training loop


In [ ]:
RESULTS = {}
ACTIVE_MODEL = ACTIVE_TRAINER = WANDB_RUN = None
LAST = {}
for ds, seed, tag in selected:
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    tr, va, te = make_datasets(ds, seed)
    bs, accum, batch_probe = resolve_batch(tag, tr)
    config = make_config(ds, seed, tr, va, te, bs, accum)
    artifacts = ft.Artifacts(LOCAL_ROOT / tag, None if SMOKE else DRIVE_DIR / tag, smoke=SMOKE)
    status = ft.run_status(artifacts, config, resume=RESUME, extend_epochs=EXTEND_EPOCHS)
    config = status['config']
    print(f'[run] {tag}: status={status["status"]} epochs={config["epochs"]}')
    LAST = {'tag': tag, 'artifacts': artifacts, 'config': config}
    if status['status'] == 'complete':
        RESULTS[tag] = {'res': status['result']}
        continue
    tag_resume = status['status'] == 'resume'
    tag_warm_start = status['warm_start'] if status['status'] == 'initialized' else (WARM_START_WEIGHTS if tag == RUN_TARGET else '')
    if tag_warm_start and not Path(tag_warm_start).is_file():
        raise FileNotFoundError('Uncommitted warm-start requires its original weights: ' + tag_warm_start)
    WANDB_RUN = ft.init_wandb('crossgate_3branch_' + tag, config, artifacts, resume=status['status'] != 'new', smoke=SMOKE, warm_start=tag_warm_start)
    log_batch_probe(WANDB_RUN, batch_probe, bs, accum)
    stopped, started = False, time.time()
    epoch_embeddings = {'z': [], 'e3d': [], 'e2d': [], 'x': [], 'y': []}
    try:
        ACTIVE_MODEL = make_model(tag_resume, tag_warm_start, artifacts)
        ACTIVE_TRAINER = ft.Trainer(ACTIVE_MODEL, tr, config, artifacts, WANDB_RUN, resume=tag_resume, warm_start=tag_warm_start, num_workers=NUM_WORKERS)
        print(f'[train] start {tag}: epochs={config["epochs"]} bs={bs} accum={accum} eff={bs * accum} workers={NUM_WORKERS} cuda={DEVICE.type == "cuda"}')
        stopped = not ACTIVE_TRAINER.fit(make_val_eval(va, bs, epoch_embeddings), test_evaluate=make_test_eval(te, bs))
        print('[train] fit finished | stopped =', stopped)
        if not stopped:
            ACTIVE_MODEL.load_state_dict(ACTIVE_TRAINER.best_state)
            RESULTS[tag] = finish_tag(ACTIVE_MODEL, ACTIVE_TRAINER, tr, va, te, artifacts, WANDB_RUN, tag, seed, started, bs)
    except BaseException:
        WANDB_RUN.finish(exit_code=1)
        raise
    if stopped:
        WANDB_RUN.summary['stopped_safely'] = True
        WANDB_RUN.finish(exit_code=0)
        print('Stopped safely. Set RESUME=True and RUN_TARGET to', tag)
        break
    ft.complete_run(artifacts, config, WANDB_RUN.id)
    ACTIVE_MODEL.cpu()
    ACTIVE_TRAINER = None
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
print('Completed:', list(RESULTS))

## Information-theory survey (optional, `RUN_INFO`)

Deep evaluation of the trained representation:

1. embeddings per branch (`resnext3d`, `view0`, `view1`) and the fused `z`;
2. probe-based branch ablation (3D only / view only / 3D+view / all / fused) with logistic + MLP probes;
3. multi-seed MI estimators (DV, NWJ, InfoNCE) + MIC with median/mean/std/min/max and negative-rate
   diagnostics (negative estimates are estimator failure, never clamped);
4. surrogate permutation testing (MIC: `IT_SURROGATES`; MINE: `IT_MINE_SURROGATES`) -> p-value + z-score;
5. joint/conditional MI and interaction information `I(Z1;Z2) - I(Z1;Z2|Y)` as redundancy (positive) vs
   synergy (negative) proxy;
6. information plane from the per-epoch embeddings captured during training.

Everything is logged to W&B as tables/scalars (`report/*_table`, `entropy/*`, `mi/*`, `surrogate/*`,
`interaction/*`) and saved to `info_theory.json` / `info_theory.pt`; no metric plots.

In [ ]:
analysis_run = None
info_results = {}
if RUN_INFO and RESULTS:
    tag = LAST['tag']
    analysis_model = (ft.SmokeModel() if SMOKE else fm.FinalModel(n_2d=N_2D, D=D_LATENT, enc2d=ENC2D, enc3d_features=ENC3D_FEATURES, enc2d_pretrained=False)).to(DEVICE)
    weights = LAST['artifacts'].local / 'best_weights.pt'
    if not ft.load_weights(analysis_model, weights):
        raise FileNotFoundError(weights)
    print('[info] analysis model:', weights)
    tr_e = it.collect_embeddings(analysis_model, tr, BS, input_res=IT_INPUT_RES, num_workers=NUM_WORKERS)
    va_e = it.collect_embeddings(analysis_model, va, BS, input_res=IT_INPUT_RES, num_workers=NUM_WORKERS)
    te_e = it.collect_embeddings(analysis_model, te, BS, input_res=IT_INPUT_RES, num_workers=NUM_WORKERS)
    print(f'[info] embeddings: train={len(tr_e["y"])} val={len(va_e["y"])} test={len(te_e["y"])}')
    def branch(capture, key):
        if key == 'view0':
            return capture['e2d'][:, 0]
        if key == 'view1':
            return capture['e2d'][:, 1]
        return capture[key]
    branches = {'z_fused': 'z', 'resnext3d': 'e3d', 'view0': 'view0'}
    if N_2D > 1:
        branches['view1'] = 'view1'
    info_results['label_entropy'] = it.label_entropy(va_e['y'])
    analysis_run = WANDB_RUN if WANDB_RUN is not None else ft.init_wandb('crossgate_3branch_' + tag, LAST['config'], LAST['artifacts'], resume=True, smoke=SMOKE)
    print('[info] branches:', list(branches), '| label entropy', round(info_results['label_entropy'], 4))
else:
    print('Information-theory analysis skipped (RUN_INFO=False or no completed run).')

### Branch representations and probe ablation
Probes are fit on training embeddings and scored on test. The ablation compares each branch alone, 3D+view
combos, the concatenation of all branches, and the fused `z` (probe-level ablation; retraining-level ablation
uses `N_2D=1` or a separate `RUN_GROUP`).

In [ ]:
if RUN_INFO and RESULTS:
    def combo_matrix(capture, keys):
        return np.concatenate([branch(capture, key) for key in keys], axis=1)
    combos = {'resnext3d': ['e3d'], 'view0': ['view0'], 'z_fused': ['z']}
    if N_2D > 1:
        combos.update({'view1': ['view1'], '3d+view0': ['e3d', 'view0'], '3d+view1': ['e3d', 'view1'], 'view0+view1': ['view0', 'view1'], 'all_concat': ['e3d', 'view0', 'view1']})
    probe_rows = []
    for name, keys in combos.items():
        for kind in ('logistic', 'mlp'):
            result = it.linear_probe(combo_matrix(tr_e, keys), tr_e['y'], combo_matrix(te_e, keys), te_e['y'], kind=kind, steps=IT_PROBE_STEPS, seed=SEEDS[0], device=str(DEVICE))
            probe_rows.append({'name': name if kind == 'logistic' else name + '_mlp', 'kind': kind, 'acc': result['acc'], 'auc': result['auc'], 'f1': result['f1']})
    info_results['probes'] = probe_rows
    probe_table = wandb.Table(columns=['representation', 'kind', 'acc', 'auc', 'f1'])
    for row in probe_rows:
        probe_table.add_data(row['name'], row['kind'], row['acc'], row['auc'], row['f1'])
    analysis_run.log({'report/probe_table': probe_table})
    analysis_run.summary.update({f'probes/{row["name"]}_{row["kind"]}_auc': row['auc'] for row in probe_rows})
    print('[probe] rows:', len(probe_rows))

### Multi-seed MI estimators + MIC
DV, NWJ and InfoNCE are trained `IT_ESTIMATOR_SEEDS` times per representation; the table reports
median/mean/std/min/max and the negative rate. Negative values indicate estimator failure/bias, not negative
information.

In [ ]:
if RUN_INFO and RESULTS:
    mi_rows = {}
    for split, capture in (('val', va_e), ('test', te_e)):
        for name, key in branches.items():
            features = branch(capture, key)
            estimates = it.estimate_mi(features, capture['y'][:, None], methods=IT_ESTIMATOR_METHODS, seeds=IT_ESTIMATOR_SEEDS, steps=IT_MINE_STEPS, hidden=IT_MINE_HIDDEN, device=str(DEVICE))
            mic = it.max_mic_features(features, capture['y'], pcs=IT_MIC_PCS)['max_mic']
            nmi = estimates['dv']['median'] / info_results['label_entropy'] if info_results['label_entropy'] > 0 else float('nan')
            mi_rows[f'{split}/{name}'] = {**estimates, 'max_mic': mic, 'nmi_dv': nmi}
            print('[mi]', split, name, {method: round(estimates[method]['median'], 3) for method in IT_ESTIMATOR_METHODS}, 'mic', round(mic, 3))
    info_results['mi'] = mi_rows
    mi_table = wandb.Table(columns=['split', 'representation', 'method', 'median', 'mean', 'std', 'min', 'max', 'negative_rate', 'max_mic', 'nmi'])
    for row_key, rep in mi_rows.items():
        split, name = row_key.split('/', 1)
        for method in IT_ESTIMATOR_METHODS:
            stats = rep[method]
            mi_table.add_data(split, name, method, stats['median'], stats['mean'], stats['std'], stats['min'], stats['max'], stats['negative_rate'], rep['max_mic'], rep['nmi_dv'])
    analysis_run.log({'report/mi_estimators_table': mi_table})
    analysis_run.log({'entropy/labels': info_results['label_entropy']})
    analysis_run.summary.update({'entropy/labels': info_results['label_entropy']})
    for row_key, rep in mi_rows.items():
        for method in IT_ESTIMATOR_METHODS:
            analysis_run.summary.update({f'mi/{row_key}/{method}_median': rep[method]['median']})

### Surrogate permutation testing
Label permutations build the null distribution; the p-value is `(1 + null >= real) / (n + 1)` and the
Bonferroni threshold is `0.05 / n_tests`.

In [ ]:
if RUN_INFO and RESULTS:
    surrogate = {}
    for name in ('z_fused', 'resnext3d'):
        features = branch(va_e, branches[name])
        surrogate[name] = it.surrogate_test(lambda a, b: it.mine_mi(a, b, steps=IT_MINE_STEPS, hidden=IT_MINE_HIDDEN, seed=SEEDS[0], device=str(DEVICE))['mi'], features, va_e['y'], n=IT_MINE_SURROGATES, seed=SEEDS[0])
    projected = it.PCA(IT_MIC_PCS).fit_transform(branch(va_e, branches['z_fused']))
    surrogate['z_fused_mic'] = it.surrogate_test(lambda a, b: it.mic_approx(a[:, 0], b)['mic'], projected, va_e['y'], n=IT_SURROGATES, seed=SEEDS[0])
    info_results['surrogate'] = surrogate
    surrogate_table = wandb.Table(columns=['statistic', 'real', 'null_mean', 'null_std', 'p_value', 'z_score', 'n'])
    for key, value in surrogate.items():
        surrogate_table.add_data(key, value['real'], value['null_mean'], value['null_std'], value['p_value'], value['z_score'], value['n'])
    analysis_run.log({'report/surrogate_table': surrogate_table})
    analysis_run.summary.update({f'surrogate/{key}_p': value['p_value'] for key, value in surrogate.items()})
    analysis_run.summary.update({'surrogate/bonferroni': 0.05 / max(len(surrogate), 1)})
    for key, value in surrogate.items():
        print('[surrogate]', key, 'p', value['p_value'], 'z', round(value['z_score'], 2))

### Redundancy vs synergy (interaction information)
`interaction = I(Z1;Z2) - I(Z1;Z2|Y)`: positive suggests redundancy, negative suggests synergy (McGill
interaction information); it is a proxy, not a full PID. Pairwise MI and joint MI are multi-seed DV medians.

In [ ]:
if RUN_INFO and RESULTS:
    pairs = [('resnext3d', 'view0')]
    if N_2D > 1:
        pairs += [('resnext3d', 'view1'), ('view0', 'view1')]
    interaction_rows = []
    for left, right in pairs:
        result = it.interaction_information(branch(va_e, branches[left]), branch(va_e, branches[right]), va_e['y'], steps=IT_MINE_STEPS, hidden=IT_MINE_HIDDEN, seeds=IT_ESTIMATOR_SEEDS, device=str(DEVICE))
        interaction_rows.append({'pair': f'{left}|{right}', 'dependence': result['dependence'], 'dependence_given_y': result['dependence_given_y'], 'interaction': result['interaction'], 'synergy_proxy': result['synergy_proxy']})
        print('[interaction]', left, right, 'dep', round(result['dependence'], 3), 'cond', round(result['dependence_given_y'], 3), 'II', round(result['interaction'], 3))
    info_results['interaction'] = interaction_rows
    interaction_table = wandb.Table(columns=['pair', 'dependence', 'dependence_given_y', 'interaction', 'synergy_proxy'])
    for row in interaction_rows:
        interaction_table.add_data(row['pair'], row['dependence'], row['dependence_given_y'], row['interaction'], row['synergy_proxy'])
    analysis_run.log({'report/interaction_table': interaction_table})
    analysis_run.summary.update({f'interaction/{row["pair"]}': row['interaction'] for row in interaction_rows})

### Information plane
`I(X;Z)` vs `I(Z;Y)` per training epoch from the captured embeddings (estimator values may be negative; read
the trend, not the absolute level).

In [ ]:
if RUN_INFO and RESULTS:
    plane = []
    if epoch_embeddings['z']:
        plane = it.information_plane(epoch_embeddings['x'][0], epoch_embeddings['z'], epoch_embeddings['y'][0], folds=IT_PLANE_FOLDS, pca_dim=IT_PCA_DIM, probe_steps=IT_PROBE_STEPS, mine_steps=IT_MINE_STEPS, seed=SEEDS[0], device=str(DEVICE), hidden=IT_MINE_HIDDEN)
    info_results['plane'] = plane
    plane_table = wandb.Table(columns=['epoch', 'mi_zy', 'mi_xz', 'probe_auc', 'probe_acc'])
    for row in plane:
        plane_table.add_data(row['epoch'], row['mi_zy'], row['mi_xz'], row['probe_auc'], row['probe_acc'])
    analysis_run.log({'report/plane_table': plane_table})
    print('[plane] rows:', len(plane))

In [ ]:
if RUN_INFO and RESULTS:
    info_results['estimator_check'] = it.validate_estimators(n=200 if SMOKE else 800, steps=60 if SMOKE else 500, seed=SEEDS[0], device=str(DEVICE))
    analysis_run.log({'estimator/xor_mi': info_results['estimator_check']['xor_mi'], 'estimator/independent_mi': info_results['estimator_check']['independent_mi']})
    analysis_run.summary.update({'estimator/xor_mi': info_results['estimator_check']['xor_mi'], 'estimator/independent_mi': info_results['estimator_check']['independent_mi']})
    LAST['artifacts'].save(info_results, 'info_theory.pt')
    json_path = LAST['artifacts'].local / 'info_theory.json'
    json_path.write_text(json.dumps(info_results, indent=2, default=float))
    LAST['artifacts'].sync(json_path)
    analysis_run.finish(exit_code=0)
    ft.complete_run(LAST['artifacts'], LAST['config'], analysis_run.id)
    print('[info] artifacts:', sorted(info_results))
else:
    if globals().get('WANDB_RUN') is not None:
        WANDB_RUN.finish(exit_code=0)
    print('Information-theory analysis skipped (RUN_INFO=False or no completed run).')

## Outputs and limitations
- `metrics.json` / `test_predictions.pt`: calibrated `train/val/test` metrics, bootstrap CI, full history.
- W&B (tables + numbers, no metric plots): `report/split_table`, `report/history_table`,
  per-step `train/*`, per-epoch `val/*` + `test/*`, calibrated `train|val|test/*` with CI,
  `xai/fusion_table` + `xai/*`, and (when `RUN_INFO`) `report/probe_table`, `report/mi_estimators_table`,
  `report/surrogate_table`, `report/interaction_table`, `report/plane_table` + `entropy/*`, `mi/*`,
  `surrogate/*`, `interaction/*` values.
- `info_theory.json` / `info_theory.pt` when `RUN_INFO=True` (no figures).
- Drive: `MasterBKDN/Thesis/crossgate_3branch/<RUN_GROUP>/<tag>/` (plus `data/` caches).
- Trains the 3D branch at 96^3 with raw 200^3 storage (2D views from raw 200^3); switching storage requires
  matching `STORE_RES`/`RES3D`/`HF_DATA_REPO` and a new `RUN_GROUP`.
- Single-seed by default; run 3-5 seeds for mean +/- std. `N_2D=1` is the ablation baseline for the number of
  2D branches. Resume only replays work since the last committed optimizer step.

In [ ]:
rows = ft.publish_summary(STORAGE)
if rows:
    print(rows)
else:
    print('No completed runs; no summary attempted.')
print('Smoke verified only local/offline behavior.' if SMOKE else 'Completed artifacts were synced to the verified Drive mount.')
